In [1]:
#!pip install flask

In [ ]:
#!pip install flask_ngrok

## SQL Server

#### Create API

In [ ]:
#!pip install pyodbc

In [1]:
# === Required Libraries ===
from flask import Flask, request, jsonify  # Flask for API, request for client data, jsonify to return JSON
import pyodbc  # ODBC driver for connecting to SQL Server
import threading  # Allows us to run Flask in a background thread (non-blocking)
import time  # Used to pause to ensure server starts

# === Initialize Flask App ===
app = Flask(__name__)  # Create an instance of the Flask web application

# === SQL Server Connection String ===
# Uses ODBC Driver 17 and connects to your local SQL Server Express instance, targeting CustomersDB
conn_str = (
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=localhost\\SQLEXPRESS;'
    'DATABASE=CustomersDB;'
    'Trusted_Connection=yes;'
)

# === Helper Function to Open Database Connection ===
def get_db_connection():
    return pyodbc.connect(conn_str)  # Establish and return a new SQL connection using the above credentials

# === GET /customers: Return all customers ===
@app.route('/customers', methods=['GET'])  # Define route for GET requests to /customers
def get_customers():
    conn = get_db_connection()  # Connect to the database
    cursor = conn.cursor()  # Create a cursor object to execute queries
    cursor.execute("SELECT CustomerID, Name, Age, Gender, Region FROM dbo.Customers")  # Query all customers
    rows = cursor.fetchall()  # Fetch all rows from the result
    conn.close()  # Close the database connection

    # Format each row as a dictionary to return JSON
    result = [
        {
            "CustomerID": row.CustomerID,
            "Name": row.Name,
            "Age": row.Age,
            "Gender": row.Gender,
            "Region": row.Region
        }
        for row in rows
    ]
    return jsonify(result)  # Return list of customers as JSON

# === POST /customers: Add a new customer ===
@app.route('/customers', methods=['POST'])  # Define route for POST requests to /customers
def add_customer():
    data = request.json  # Parse incoming JSON payload
    try:
        conn = get_db_connection()  # Open DB connection
        cursor = conn.cursor()  # Create cursor
        # Insert new customer with provided fields
        cursor.execute("""
            INSERT INTO dbo.Customers (CustomerID, Name, Age, Gender, Region)
            VALUES (?, ?, ?, ?, ?)
        """, data['CustomerID'], data['Name'], data['Age'], data['Gender'], data['Region'])
        conn.commit()  # Save changes
        return jsonify({'message': 'Customer added'}), 201  # Return success message
    except Exception as e:
        return jsonify({'error': str(e)}), 400  # Return error message if something goes wrong
    finally:
        conn.close()  # Always close connection

# === PUT /customers/<id>: Update customer by ID ===
@app.route('/customers/<int:id>', methods=['PUT'])  # Define route for PUT requests with customer ID
def update_customer(id):
    data = request.json  # Parse JSON from request
    try:
        conn = get_db_connection()  # Open DB connection
        cursor = conn.cursor()  # Create cursor
        # Update customer details using parameterized SQL
        cursor.execute("""
            UPDATE dbo.Customers
            SET Name = ?, Age = ?, Gender = ?, Region = ?
            WHERE CustomerID = ?
        """, data['Name'], data['Age'], data['Gender'], data['Region'], id)
        conn.commit()  # Commit update
        return jsonify({'message': 'Customer updated'})  # Return success message
    except Exception as e:
        return jsonify({'error': str(e)}), 400  # Return error on failure
    finally:
        conn.close()  # Close DB connection

# === DELETE /customers/<id>: Delete customer by ID ===
@app.route('/customers/<int:id>', methods=['DELETE'])  # Define route for DELETE requests with customer ID
def delete_customer(id):
    try:
        conn = get_db_connection()  # Open DB connection
        cursor = conn.cursor()  # Create cursor
        cursor.execute("DELETE FROM dbo.Customers WHERE CustomerID = ?", id)  # Delete customer by ID
        conn.commit()  # Commit changes
        return jsonify({'message': 'Customer deleted'})  # Return success
    except Exception as e:
        return jsonify({'error': str(e)}), 400  # Return error if deletion fails
    finally:
        conn.close()  # Always close connection

# === Run Flask App in a Background Thread (for use in Jupyter) ===
def run_flask():
    app.run(port=5000, debug=False, use_reloader=False)  # Run Flask server on port 5000 without reloading

threading.Thread(target=run_flask).start()  # Start the Flask app in a separate thread (so Jupyter doesn't hang)
time.sleep(1)  # Wait a second to let the server start before making requests

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [05/Aug/2025 11:28:05] "POST /customers HTTP/1.1" 201 -
127.0.0.1 - - [05/Aug/2025 11:28:06] "GET /customers HTTP/1.1" 200 -
127.0.0.1 - - [05/Aug/2025 11:28:07] "PUT /customers/101 HTTP/1.1" 200 -
127.0.0.1 - - [05/Aug/2025 11:28:08] "DELETE /customers/101 HTTP/1.1" 200 -
127.0.0.1 - - [05/Aug/2025 11:41:02] "GET /customers HTTP/1.1" 200 -
127.0.0.1 - - [05/Aug/2025 11:41:02] "GET /favicon.ico HTTP/1.1" 404 -


#### Consume(Call) API

In [2]:
import requests

In [3]:
# 🧪 POST: Add customer
res = requests.post("http://127.0.0.1:5000/customers", json={
    "CustomerID": 101,
    "Name": "Alice Smith",
    "Age": 30,
    "Gender": "Female",
    "Region": "North"
})
print("POST:", res.json())

POST: {'message': 'Customer added'}


In [4]:
# 📥 GET all customers
res = requests.get("http://127.0.0.1:5000/customers")
print("GET:", res.json())

GET: [{'Age': 30, 'CustomerID': 1, 'Gender': 'Male', 'Name': 'Ahmed Hassan', 'Region': 'Cairo'}, {'Age': 42, 'CustomerID': 2, 'Gender': 'Male', 'Name': 'Mohamed Tarek', 'Region': 'Alexandria'}, {'Age': 28, 'CustomerID': 3, 'Gender': 'Female', 'Name': 'Heba Ali', 'Region': 'Giza'}, {'Age': 35, 'CustomerID': 4, 'Gender': 'Female', 'Name': 'Salma Nabil', 'Region': 'Mansoura'}, {'Age': 39, 'CustomerID': 5, 'Gender': 'Male', 'Name': 'Youssef Adel', 'Region': 'Cairo'}, {'Age': 25, 'CustomerID': 6, 'Gender': 'Female', 'Name': 'Nourhan Samir', 'Region': 'Ismailia'}, {'Age': 30, 'CustomerID': 101, 'Gender': 'Female', 'Name': 'Alice Smith', 'Region': 'North'}]


In [5]:
# ✏️ PUT: Update customer
res = requests.put("http://127.0.0.1:5000/customers/101", json={
    "Name": "Alice Updated",
    "Age": 31,
    "Gender": "Female",
    "Region": "South"
})
print("PUT:", res.json())

PUT: {'message': 'Customer updated'}


In [6]:
# ❌ DELETE: Remove customer
res = requests.delete("http://127.0.0.1:5000/customers/101")
print("DELETE:", res.json())

DELETE: {'message': 'Customer deleted'}
